# Treino da segmentação de corais na GPU do Colab

Treina o modelo de segmentação de corais em GPU, reproduzindo o mesmo pipeline que roda localmente em CPU.

O treino local de 30 épocas levou 3h30 em CPU e o resultado ficou subtreinado: o mAP50 da máscara ainda subia quando o cronograma terminou. Em uma T4 o mesmo trabalho leva minutos, o que torna viável treinar as 130 épocas configuradas aqui em cerca de 2 horas.

**Antes de começar:** em `Ambiente de execução > Alterar o tipo de ambiente de execução`, selecione **GPU (T4)**.

O dataset não precisa ser enviado. O Coralscapes é baixado direto do Hugging Face pela rede do Google, o que leva poucos minutos.

Atenção aos limites do plano gratuito: a sessão cai por inatividade após cerca de 90 minutos e tem duração máxima limitada. Os pesos são salvos no Google Drive ao final para não se perderem quando a máquina for reciclada.

In [ ]:
import torch

print("GPU disponivel:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "Sem GPU. Va em Ambiente de execucao > Alterar o tipo de ambiente "
        "de execucao e selecione GPU."
    )

nome = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{nome} com {vram:.1f} GB")
print("torch", torch.__version__)

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

DESTINO_DRIVE = "/content/drive/MyDrive/coral_treino"
os.makedirs(DESTINO_DRIVE, exist_ok=True)

REPO = "https://github.com/AI-SPARC/subdrone-image-processing.git"
if not os.path.isdir("/content/subdrone-image-processing"):
    !git clone --depth 1 {REPO} /content/subdrone-image-processing

PROJETO = "/content/subdrone-image-processing/segmentation/corais"
os.chdir(PROJETO)
print("diretorio:", os.getcwd())
print(sorted(f for f in os.listdir() if f.endswith((".py", ".yaml", ".json"))))

In [ ]:
!pip install -q ultralytics datasets

import ultralytics
ultralytics.checks()

## Dataset

Baixa o Coralscapes e exporta imagens e máscaras, depois converte as máscaras indexadas em polígonos YOLO-seg. O download bruto tem cerca de 6 GB e fica no cache do Hugging Face; a exportação reduz as imagens para 1024 px de largura, já que o treino usa 640.

A conversão agrupa as 39 classes originais nas 3 do projeto conforme `map_coral.json`. O passo de validação confere que nenhum rótulo saiu como caixa em vez de polígono e que as coordenadas estão normalizadas.

In [ ]:
!python exportar_coralscapes.py --max-width 1024

for split in ["train", "valid", "test"]:
    !python convert_annotations.py mask2yolo --masks {split}/masks --out {split}/labels --class-map map_coral.json --min-area 1500

!python convert_annotations.py validate --labels train/labels --nc 3
!python convert_annotations.py stats --labels train/labels

## Treino

Duas fases, como no pipeline local. A primeira congela o backbone e ajusta apenas a cabeça da rede; a segunda descongela tudo com taxa de aprendizado menor.

Os scripts detectam a GPU sozinhos e escolhem lote 16 e precisão mista. As épocas aqui são bem maiores que as usadas em CPU, que era a limitação real do treino local.

Se a sessão cair no meio da fase 2, os pesos de cada época ficam em `runs/segment/`, e é possível retomar apontando `--peso` para o último checkpoint.

In [ ]:
!python train_phase1.py --epochs 30 --workers 2

In [ ]:
!python train_phase2.py --epochs 100 --workers 2

## Avaliação

Mede o modelo final no split de teste, que não foi usado em nenhum momento do treino.

O número a observar não é o `all`, e sim a linha de cada classe. No treino em CPU o recall da máscara ficou em 0,551 para `coral_vivo`, 0,218 para `coral_branqueado` e 0,090 para `coral_morto`. Essa última é a que mais importa acompanhar: se continuar próxima de zero com o treino mais longo, o problema não é falta de épocas, e sim ambiguidade visual da classe.

In [ ]:
!python evaluate.py val \
    --model trained_models/yolo_coral_seg/best.pt \
    --data data.yaml --split test

In [ ]:
import glob
import shutil

copiados = []

for peso in glob.glob("trained_models/yolo_coral_seg/*"):
    shutil.copy(peso, DESTINO_DRIVE)
    copiados.append(os.path.basename(peso))

# results.csv e os graficos sustentam a analise depois que a sessao morrer.
# O treino fica em runs/segment/<projeto>/<run>; a validacao, em runs/segment/<run>.
for run in glob.glob("runs/segment/*") + glob.glob("runs/segment/*/*"):
    if not os.path.isdir(run):
        continue
    arquivos = []
    for padrao in ("results.csv", "args.yaml", "*.png", "*.jpg"):
        arquivos += glob.glob(os.path.join(run, padrao))
    if not arquivos:
        continue
    alvo = os.path.join(DESTINO_DRIVE, os.path.basename(run))
    os.makedirs(alvo, exist_ok=True)
    for arq in arquivos:
        shutil.copy(arq, alvo)
        copiados.append(os.path.relpath(arq, "runs/segment"))

print(f"{len(copiados)} arquivos copiados para {DESTINO_DRIVE}")
for c in copiados:
    print(" ", c)